In [4]:
import pandas as pd
import re
from pathlib import Path

pd.set_option("display.max_colwidth", 200)


In [5]:
df_raw = pd.read_csv("../data/processed/messages_raw.csv")

print("Shape:", df_raw.shape)
df_raw.head()


Shape: (4592, 24)


,id,type,date,date_unixtime,from,from_id,via_bot,photo,photo_file_size,width,...,edited,edited_unixtime,reactions,reply_to_message_id,poll.question,poll.closed,poll.total_voters,poll.answers,forwarded_from,text_length
0,12,message,2021-09-05T00:24:19,1630785259,معرفی اساتید علموص,channel1558862150,@chToolsBot,(File not included. Change data exporting settings to download.),33460.0,640.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,679
1,14,message,2021-09-05T00:34:39,1630785879,معرفی اساتید علموص,channel1558862150,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,908
2,15,message,2021-09-05T00:35:51,1630785951,معرفی اساتید علموص,channel1558862150,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1140
3,16,message,2021-09-05T00:36:56,1630786016,معرفی اساتید علموص,channel1558862150,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,941
4,17,message,2021-09-05T00:37:00,1630786020,معرفی اساتید علموص,channel1558862150,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,889


In [6]:
df = df_raw.copy()

df = df[
    (df["type"] == "message") &
    (df["text"].notna())
].reset_index(drop=True)

print("After filtering:", df.shape)


After filtering: (4556, 24)


In [7]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [8]:
def parse_message(text: str) -> dict:
    result = {
        "professor_name_raw": None,
        "department": None,
        "course_name": None,
        "rating_1": None,
        "rating_2": None,
        "rating_3": None,
        "rating_4": None,
        "rating_5": None,
        "rating_6": None,
        "grading_status_raw": None,
        "attendance_status_raw": None,
        "comment_text": None,
        "parse_error": False
    }

    try:
        text = normalize_text(text)

        # اسم استاد
        prof_match = re.search(r"(?:استاد|دکتر)\s*[:：]?\s*(.+)", text)
        if prof_match:
            result["professor_name_raw"] = prof_match.group(1).split("\n")[0].strip()

        # دانشکده
        dept_match = re.search(r"دانشکده\s*[:：]?\s*(.+)", text)
        if dept_match:
            result["department"] = dept_match.group(1).split("\n")[0].strip()

        # اسم درس
        course_match = re.search(r"درس\s*[:：]?\s*(.+)", text)
        if course_match:
            result["course_name"] = course_match.group(1).split("\n")[0].strip()

        # امتیازها (۶ عدد از ۱۰)
        ratings = re.findall(r"(\d{1,2})\s*/\s*10", text)
        ratings = ratings[:6]
        for i, r in enumerate(ratings):
            result[f"rating_{i+1}"] = int(r)

        # نمره‌دهی
        if re.search(r"سخت", text):
            result["grading_status_raw"] = "سختگیر"
        elif re.search(r"منصف", text):
            result["grading_status_raw"] = "منصفانه"
        elif re.search(r"آسان|راحت", text):
            result["grading_status_raw"] = "آسان"

        # حضور و غیاب
        if re.search(r"حضور.?غیاب.*سخت", text):
            result["attendance_status_raw"] = "سختگیر"
        elif re.search(r"حضور.?غیاب.*آزاد", text):
            result["attendance_status_raw"] = "آزاد"

        # کامنت آزاد (فرض: بعد از خط «نظر»)
        comment_match = re.search(r"نظر\s*[:：]?\s*(.+)", text, re.S)
        if comment_match:
            result["comment_text"] = comment_match.group(1).strip()

    except Exception:
        result["parse_error"] = True

    return result


In [9]:
parsed_rows = []

for _, row in df.iterrows():
    parsed = parse_message(row["text"])
    parsed_rows.append(parsed)

df_parsed = pd.DataFrame(parsed_rows)

print("Parsed shape:", df_parsed.shape)
df_parsed.head()


Parsed shape: (4556, 13)


,professor_name_raw,department,course_name,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,grading_status_raw,attendance_status_raw,comment_text,parse_error
0,None,None,"ی'}, ' ، ', {'type': 'hashtag', 'text': '#حضور_غیاب'}, ' ، ', {'type': 'hashtag', 'text': '#نمره_دهی'}, ' و ... رو میتونین از این کانال بدست بیارید.\n\n🆔 ', {'type': 'mention', 'text': '@ostad_elm...",None,None,None,None,None,None,None,None,None,False
1,"کلاس داشته:\n ┘ مهر 99\n\nتوضیحات:\n ┘ چیزی اضافه ایی نیست\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'text': '@ostad_elmosiBot'}, '\n\nکانال معرفی اسات...",None,7\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 8\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ چیزی ندارم\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ مهر ...,None,None,None,None,None,None,منصفانه,None,None,False
2,"هاجر فلاحتی\n🏫 ', {'type': 'hashtag', 'text': '#مهندسی_کامپیوتر'}, '\n📒 مدار منطقی- طراحی سیستم های کامپیوتری\n\nمنابع آموزش\n ┘ یادم نمیاد\n\nحضور و غیاب\n ┘ حضور مهم نیست اما تاثیر مثبت دارد\n\n...",None,1\n ┤ نحوه مدیریت کلاس(نظم و زمان): 3\n ┤ پاسخگویی(حضوری و غیرحضوری): 2\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ چیزی ندارم\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ بهمن...,None,None,None,None,None,None,None,None,None,False
3,"کلاس داشته:\n ┘ بهمن 99\n\nتوضیحات:\n ┘ در کل اگر که دنبال یک استاد با ادب با دانشجو میخواید گزینه خوبیه\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'tex...",None,10\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 10\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ بله\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ بهمن 99\n...,None,None,None,None,None,None,منصفانه,None,None,False
4,"کلاس داشته:\n ┘ مهر 98\n\nتوضیحات:\n ┘ سخت گیر و پربازده\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'text': '@ostad_elmosiBot'}, '\n\nکانال معرفی اساتید...",None,9\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 9\n ┤ آداب و رفتار اجتماعی با دانشجویان: 9\n\nراه ارتباطی:\n ┘ شماره تماس\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ مهر 9...,None,None,None,None,None,None,سختگیر,None,None,False


In [10]:
df_parsed.isna().mean().sort_values(ascending=False)


rating_1                 1.000000
rating_2                 1.000000
rating_3                 1.000000
rating_4                 1.000000
rating_5                 1.000000
rating_6                 1.000000
attendance_status_raw    0.984416
department               0.971905
comment_text             0.890255
grading_status_raw       0.505707
course_name              0.057726
professor_name_raw       0.017779
parse_error              0.000000
dtype: float64

In [11]:
df_parsed["parse_error"].value_counts()


parse_error
False    4556
Name: count, dtype: int64

In [12]:
df_final = pd.concat(
    [
        df[["id", "date", "date_unixtime"]],
        df_parsed
    ],
    axis=1
)

df_final.head()


,id,date,date_unixtime,professor_name_raw,department,course_name,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,grading_status_raw,attendance_status_raw,comment_text,parse_error
0,12,2021-09-05T00:24:19,1630785259,None,None,"ی'}, ' ، ', {'type': 'hashtag', 'text': '#حضور_غیاب'}, ' ، ', {'type': 'hashtag', 'text': '#نمره_دهی'}, ' و ... رو میتونین از این کانال بدست بیارید.\n\n🆔 ', {'type': 'mention', 'text': '@ostad_elm...",None,None,None,None,None,None,None,None,None,False
1,14,2021-09-05T00:34:39,1630785879,"کلاس داشته:\n ┘ مهر 99\n\nتوضیحات:\n ┘ چیزی اضافه ایی نیست\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'text': '@ostad_elmosiBot'}, '\n\nکانال معرفی اسات...",None,7\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 8\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ چیزی ندارم\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ مهر ...,None,None,None,None,None,None,منصفانه,None,None,False
2,15,2021-09-05T00:35:51,1630785951,"هاجر فلاحتی\n🏫 ', {'type': 'hashtag', 'text': '#مهندسی_کامپیوتر'}, '\n📒 مدار منطقی- طراحی سیستم های کامپیوتری\n\nمنابع آموزش\n ┘ یادم نمیاد\n\nحضور و غیاب\n ┘ حضور مهم نیست اما تاثیر مثبت دارد\n\n...",None,1\n ┤ نحوه مدیریت کلاس(نظم و زمان): 3\n ┤ پاسخگویی(حضوری و غیرحضوری): 2\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ چیزی ندارم\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ بهمن...,None,None,None,None,None,None,None,None,None,False
3,16,2021-09-05T00:36:56,1630786016,"کلاس داشته:\n ┘ بهمن 99\n\nتوضیحات:\n ┘ در کل اگر که دنبال یک استاد با ادب با دانشجو میخواید گزینه خوبیه\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'tex...",None,10\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 10\n ┤ آداب و رفتار اجتماعی با دانشجویان: 10\n\nراه ارتباطی:\n ┘ بله\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ بهمن 99\n...,None,None,None,None,None,None,منصفانه,None,None,False
4,17,2021-09-05T00:37:00,1630786020,"کلاس داشته:\n ┘ مهر 98\n\nتوضیحات:\n ┘ سخت گیر و پربازده\n~~~~~~~~~~~~~~~~~\nبرای ثبت معرفی استاد به ربات زیر پیام بدید\n', {'type': 'mention', 'text': '@ostad_elmosiBot'}, '\n\nکانال معرفی اساتید...",None,9\n ┤ نحوه مدیریت کلاس(نظم و زمان): 9\n ┤ پاسخگویی(حضوری و غیرحضوری): 9\n ┤ آداب و رفتار اجتماعی با دانشجویان: 9\n\nراه ارتباطی:\n ┘ شماره تماس\n\nترمی که دانشجو با این استاد کلاس داشته:\n ┘ مهر 9...,None,None,None,None,None,None,سختگیر,None,None,False


In [13]:
Path("../data/processed").mkdir(exist_ok=True)

df_final.to_csv(
    "../data/processed/messages_parsed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ messages_parsed.csv saved successfully")


✅ messages_parsed.csv saved successfully
